# Keep a document revision ledger

## Goal

Retain all observed versions within a cutoff, distinguishing first observation, changed content and unchanged content. No sentiment or truth classification is performed.

This notebook uses synthetic teaching data, not a paper replication or production observations.

## Setup

Use a Python 3.10+ kernel and run all cells in order. Computation uses only the standard library, without keys, networking or extra data files. Open in an existing Jupyter environment.

Embedded inputs match inputs.json in the same download directory. Edit args in the next cell to experiment; preserve explicit times and units.

In [ ]:
import json

# Synthetic inputs; no credentials or network access.
bundle = json.loads("{\"version\":1,\"tutorial\":\"document-version-ledger\",\"identity\":\"synthetic\",\"args\":[[{\"publisher\":\"DEMO\",\"documentId\":\"DOC-1\",\"version\":\"v1\",\"contentHash\":\"aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa\",\"publishedAt\":\"2025-01-06T08:00:00Z\",\"firstSeenAt\":\"2025-01-06T08:01:00Z\"},{\"publisher\":\"DEMO\",\"documentId\":\"DOC-1\",\"version\":\"v1\",\"contentHash\":\"aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa\",\"publishedAt\":\"2025-01-06T08:00:00Z\",\"firstSeenAt\":\"2025-01-06T08:01:00Z\"},{\"publisher\":\"DEMO\",\"documentId\":\"DOC-1\",\"version\":\"v2\",\"contentHash\":\"bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb\",\"publishedAt\":\"2025-01-07T08:00:00Z\",\"firstSeenAt\":\"2025-01-07T08:01:00Z\"},{\"publisher\":\"DEMO\",\"documentId\":\"DOC-1\",\"version\":\"v3\",\"contentHash\":\"bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb\",\"publishedAt\":\"2025-01-08T08:00:00Z\",\"firstSeenAt\":\"2025-01-08T08:01:00Z\"}],\"2025-01-08T09:00:00Z\"],\"expected\":[{\"publisher\":\"DEMO\",\"documentId\":\"DOC-1\",\"version\":\"v1\",\"contentHash\":\"aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa\",\"publishedAt\":\"2025-01-06T08:00:00Z\",\"firstSeenAt\":\"2025-01-06T08:01:00Z\",\"availableAt\":\"2025-01-06T08:01:00.000Z\",\"status\":\"first_observation\"},{\"publisher\":\"DEMO\",\"documentId\":\"DOC-1\",\"version\":\"v2\",\"contentHash\":\"bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb\",\"publishedAt\":\"2025-01-07T08:00:00Z\",\"firstSeenAt\":\"2025-01-07T08:01:00Z\",\"availableAt\":\"2025-01-07T08:01:00.000Z\",\"status\":\"changed_content\"},{\"publisher\":\"DEMO\",\"documentId\":\"DOC-1\",\"version\":\"v3\",\"contentHash\":\"bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb\",\"publishedAt\":\"2025-01-08T08:00:00Z\",\"firstSeenAt\":\"2025-01-08T08:01:00Z\",\"availableAt\":\"2025-01-08T08:01:00.000Z\",\"status\":\"unchanged_content\"}]}")
args = bundle["args"]
expected = bundle["expected"]
print(json.dumps(args, ensure_ascii=False, indent=2))

## Steps

### 1. Define document and version identity

Use a stable document identity within each publisher and a traceable version. Matching titles do not prove document identity; identical content at different publishers retains separate identities. Syndication clustering and entity recognition are outside the example. Retain real URLs, company IDs and source files separately.

### 2. Freeze the hashing convention

Hash a consistently defined byte stream or normalized text, recording the algorithm and parser version. Mixing conventions can make metadata or parser changes resemble revisions. The sample's 64-character strings are synthetic placeholders; the function neither downloads files nor validates actual source integrity.

### 3. Check conflicts before availability

Collapse exact repeats, but reject conflicting hashes or time evidence for the same version. Filter by the later of publication and first observation, so later acquisitions are not fictional historical holdings. Date-only sources need external review or a disclosed conservative rule, not silently assigned midnight.

### 4. Keep the change trail

Order each document's versions by availability and compare hashes with the preceding available version. Mark first observation, changed content or unchanged content; reject ambiguous equal-time revisions. A changed hash neither identifies the truer version nor implies adverse news. Interpret the actual edit separately.

### Method and assumptions

- A revision ledger needs retained historical evidence; a latest-only API cannot manufacture it.
- Equal hashes do not imply equal rights, publishers or document identities.
- Real licensing and full-text access require separate checks.

In [ ]:
from datetime import datetime, timezone
import json
import re


def preserve_document_versions(rows, cutoff):
    def parse(value):
        try:
            parsed = datetime.strptime(value, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
            if parsed.strftime("%Y-%m-%dT%H:%M:%SZ") != value:
                raise ValueError()
            return parsed
        except (ValueError, TypeError):
            raise ValueError("utc_seconds_required")

    boundary, identities, eligible = parse(cutoff), {}, []
    for row in rows:
        if not all(isinstance(row.get(key), str) and row[key] for key in ("publisher", "documentId", "version")) or not isinstance(row.get("contentHash"), str) or not re.fullmatch(r"[a-f0-9]{64}", row["contentHash"]):
            raise ValueError("invalid_document_identity")
        available = max(parse(row.get("publishedAt")), parse(row.get("firstSeenAt")))
        key = (row["publisher"], row["documentId"], row["version"])
        fingerprint = (row["contentHash"], row["publishedAt"], row["firstSeenAt"])
        if key in identities:
            if identities[key] != fingerprint:
                raise ValueError("conflicting_document_version")
            continue
        identities[key] = fingerprint
        if available <= boundary:
            eligible.append({**row, "availableAt": available.isoformat(timespec="milliseconds").replace("+00:00", "Z")})
    eligible.sort(key=lambda row: (row["availableAt"], json.dumps([row["publisher"], row["documentId"], row["version"]], ensure_ascii=False, separators=(",", ":"))))
    previous, result = {}, []
    for row in eligible:
        key = (row["publisher"], row["documentId"])
        prior = previous.get(key)
        if prior and prior["availableAt"] == row["availableAt"]:
            raise ValueError("ambiguous_revision_order")
        previous[key] = row
        status = "first_observation" if not prior else "unchanged_content" if prior["contentHash"] == row["contentHash"] else "changed_content"
        result.append({**row, "status": status})
    return result


### Run the sample

Four inputs become three versions: v1 first observation, v2 changed content, v3 unchanged content. Older versions are not overwritten.

In [ ]:
result = preserve_document_versions(*args)
print(json.dumps(result, ensure_ascii=False, indent=2))

## Checks

Compare every row with the browser example's expected output. After editing inputs, a failed assertion may be expected: explain the difference before changing the check.

In [ ]:
assert result == expected, "Output differs from the reference synthetic example"
assert bundle["identity"] == "synthetic"
print("Passed: output matches the synthetic browser example.")

## Next steps

Before real data, confirm grants, fields, schema_major, windows and provenance using authenticated GET /v1/catalog, then map the actual contract. Candidate IDs below do not guarantee availability or historical completeness. API as_of is not a historical filing-version guarantee. Validate again after substituting real inputs; the synthetic pass does not transfer.

- `cn.dataset.anns_d`
- `cn.news.flash`

### References

- [Tushare: announcement fields and source links](https://tushare.pro/document/2?doc_id=176)
- [Loughran & McDonald: parsing boundaries](https://www.uts.edu.au/globalassets/sites/default/files/adg_cons2015_loughran-mcdonald-je-2011.pdf)

[Back to tutorial](https://tradingdatas.com/recipes/document-version-ledger/)